# Weak-lensing MAF metrics (WeakLensingNvisits, RIZDetectionCoaddExposureTime) on the dust-footprint variants

- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-20
- Kernel: conda_py313_opsim53
- Context: SCOC footprint-shrinking study + DESC static-probes metrics. The goal of the series `10_DESCMAFDEPTHANDGALCOUNT` is to understand in detail some `rubin_sim` MAF metrics in order to improve them later (PSF handling, weak-lensing metric).
- This notebook: **03** of the series - the DESC **Weak Lensing (WL)** systematics-mitigation proxy metrics **`WeakLensingNvisits`** and **`RIZDetectionCoaddExposureTime`**, recomputed on each dust-threshold footprint variant of the v5.3.6 simulations: Healpix maps, histograms, and maps of the differences between consecutive thresholds.
- Companion notebooks: `01_compareExgalM5withCuts.ipynb` (`ExgalM5WithCuts`, whose dust and depth cuts are also used by `WeakLensingNvisits`) and `02_compareGalaxyCounts.ipynb`. The same WL metrics on the baseline only, year by year: `../06_MAF_DESC_TaskF/02_WL_DESC_TaskForce_demo.ipynb`.
- Presentation model: `../07_variateEVmV/01_FOMNv_HealpixDiff_ShrinkFPDust.ipynb`
- OpSim simulations analyzed (`/Users/dagoret/DATA/OpSim/`):
  - `shrink_fp_dust_0.050_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.080_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.120_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.150_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.199_v5.3.6_10yrs.db`
  - `baseline_v5.3.6_11yrs.db` (E(B-V) threshold 0.200)
  - `shrink_fp_dust_0.250_v5.3.6_10yrs.db`

## Notebook overview

**What the two metrics compute.** Both are per-pixel (`HealpixSlicer`) *proxy* metrics for weak-lensing systematics mitigation (higher is better); they do not forecast a cosmological constraint. As described in `../06_MAF_DESC_TaskF/02_WL_DESC_TaskForce_demo.ipynb`:

1. **`WeakLensingNvisits`**: the number of good visits (exposure time above `min_exp_time`) in a band combination (`gri` or `riz`) falling in the pixel, after the same Galactic-extinction and coadded-depth cuts as `ExgalM5WithCuts` (notebook 01): the pixel is rejected if `E(B-V) > ebvlim` or if the coadded `i`-band depth is below `depth_cut`.
2. **`RIZDetectionCoaddExposureTime`**: the summed exposure time of the `riz` visits in the pixel (after a dust cut and a minimum-exposure-time cut on the individual visits), a proxy for the depth of the `riz` coadd used for object detection.

The exact definitions are printed in Section 3 from the installed `rubin_sim` (signature, docstring and `run` source); they prevail over the summary above.

**Metrics computed for each of the 7 runs** (`KINDS` in Section 2):

| kind | metric | bands of the SQL selection |
|---|---|---|
| `NV_gri` | `WeakLensingNvisits` (headline WL metric of the official `science_radar_batch`) | `gri` |
| `NV_riz` | `WeakLensingNvisits` | `riz` |
| `EXPT` | `RIZDetectionCoaddExposureTime` (`det_bands = riz`) | same selection as in the 06 demo notebook (`gri`), see the caveats |

**Choices made in this notebook (all editable in Section 2).**
- The metric-side dust cut (`ebvlim`) is **adapted to each run**: it is set to the E(B-V) threshold that defines the WFD (Wide Fast Deep) footprint of that run (0.050, 0.080, 0.120, 0.150, 0.199, 0.200 for the baseline, 0.250), as in notebooks 01 and 02. Set `EBV_CUT_MODE = 'fixed'` to use the same cut `FIXED_LIM_EBV = 0.2` for all runs. The differences between consecutive maps combine two effects: the change of the metric footprint (dust cut) and the change of the visit distribution produced by the scheduler.
- `depth_cut = 25.9` (year-10 value of the official batch: 26.0 minus an offset of 0.1, as in notebook 01), `min_exp_time = 15` s, `nside = 64`, `i` band for the depth cut.
- All runs are truncated to their **first 10 years** (`night <= 10*365.25 + 0.5`), non-DDF visits only (`scheduler_note not like 'DD%'`). The full 10-year survey is the "year 10" case of the official per-year batch; the per-year evolution of the baseline is in `06_MAF_DESC_TaskF/02_WL_DESC_TaskForce_demo.ipynb`.

**How the figures are built** (same as notebooks 01 and 02).
1. Healpix map of each metric for each of the 7 runs, on one common color scale.
2. Area-weighted histograms of the valid pixels, and the cumulative area above a given value.
3. Difference maps between **consecutive** thresholds (`0.080-0.050`, `0.120-0.080`, `0.150-0.120`, `0.199-0.150`, `baseline-0.199`, `0.250-baseline`). As for the galaxy counts of notebook 02, a masked pixel counts as **0** (0 visits, 0 s), so a difference map also contains what is gained or lost with the footprint; the table splits the change of the total into *common pixels*, *gained pixels* and *lost pixels*. The pixels gained or lost are also shown in their own map.
4. Totals and footprint area as a function of the dust threshold, and the mean and median `WeakLensingNvisits` with the pixel-to-pixel dispersion.

**Caching.** MAF is run only once per (metric, simulation, E(B-V) cut); the maps are saved as `.npz` in `data_03_WL/` and reloaded at the next execution (set `FORCE_RECOMPUTE = True` to redo them).

## 1. Imports

In [ ]:
import os
import inspect
from os.path import join, isfile

import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import rubin_sim
import rubin_sim.maf as maf
import rubin_sim.maf.slicers as slicers
import rubin_sim.maf.maps as maf_maps
import rubin_sim.maf.metric_bundles as mb
from rubin_sim.maf.metrics.weak_lensing_systematics_metric import (
    WeakLensingNvisits,
    RIZDetectionCoaddExposureTime,
)

print("rubin_sim version:", rubin_sim.__version__)

## 2. Configuration

In [ ]:
NB_TAG = "WL"
data_dir = f"data_03_{NB_TAG}"
figs_dir = f"figs_03_{NB_TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
print("MAF output (data) directory :", os.path.abspath(data_dir))
print("Figures output directory    :", os.path.abspath(figs_dir))

resultsDb = maf.db.ResultsDb(out_dir=data_dir)

In [ ]:
# Directory holding the OpSim databases
OPSIM_DIR = "/Users/dagoret/DATA/OpSim"

# (run_name, E(B-V) threshold that defines the footprint of the run), sorted by increasing threshold
RUNS_INFO = [
    ("shrink_fp_dust_0.050_v5.3.6_10yrs", 0.050),
    ("shrink_fp_dust_0.080_v5.3.6_10yrs", 0.080),
    ("shrink_fp_dust_0.120_v5.3.6_10yrs", 0.120),
    ("shrink_fp_dust_0.150_v5.3.6_10yrs", 0.150),
    ("shrink_fp_dust_0.199_v5.3.6_10yrs", 0.199),
    ("baseline_v5.3.6_11yrs", 0.200),
    ("shrink_fp_dust_0.250_v5.3.6_10yrs", 0.250),
]
RUN_NAMES = [r for r, _ in RUNS_INFO]
DUST_THRESH = dict(RUNS_INFO)
REF_RUN = "baseline_v5.3.6_11yrs"

# Consecutive-threshold pairs (later run minus earlier run):
# 0.080-0.050, 0.120-0.080, 0.150-0.120, 0.199-0.150, baseline-0.199, 0.250-baseline
PAIRS = [(RUN_NAMES[i + 1], RUN_NAMES[i]) for i in range(len(RUN_NAMES) - 1)]


def get_db_path(run_name):
    # Look in OPSIM_DIR, then in OPSIM_DIR/sim_baseline (where the baseline was stored in the 06/07 notebooks)
    fname = run_name + ".db"
    candidates = [join(OPSIM_DIR, fname), join(OPSIM_DIR, "sim_baseline", fname)]
    for c in candidates:
        if isfile(c):
            return c
    raise FileNotFoundError(f"OpSim db not found for run {run_name}. Tried: {candidates}")


for run_name, dust in RUNS_INFO:
    try:
        path = get_db_path(run_name)
    except FileNotFoundError:
        path = "NOT FOUND (only a problem if there is no cached map, see section 5)"
    print(f"{run_name:40s} E(B-V) < {dust:.3f}  ->  {path}")

In [ ]:
BAND = "i"  # band of the coadded-depth cut
DEPTH_CUT = (
    25.9  # i-band coadded depth cut (year-10 value of science_radar_batch: 26.0 - 0.1), as in notebook 01
)
MIN_EXP_TIME = 15  # minimum exposure time of a visit [s], as in the official batch
NSIDE = 64  # same nside as science_radar_batch and as notebook 01
MAX_NIGHT = 10 * 365.25 + 0.5  # keep the first 10 years of every run
INFO_SUFFIX = "10yr nonDD"
FORCE_RECOMPUTE = False  # True -> ignore the cached .npz maps and rerun MAF

# E(B-V) cut applied by the METRICS (ebvlim of WeakLensingNvisits and RIZDetectionCoaddExposureTime), see metric_ebv_cut():
#   'run'   -> E(B-V) threshold that defines the WFD footprint of each run (DUST_THRESH), adapted run by run
#   'fixed' -> the same value FIXED_LIM_EBV for all runs
EBV_CUT_MODE = "run"
FIXED_LIM_EBV = 0.2

# WL maps computed for every run: kind -> label, unit and bands of the SQL selection.
# NV_*  : WeakLensingNvisits (as in the official batch: the SQL selects the bands that are counted)
# EXPT  : RIZDetectionCoaddExposureTime, det_bands = riz; the SQL bands are those used for it in
#         06_MAF_DESC_TaskF/02_WL_DESC_TaskForce_demo.ipynb (check the caveats if this map looks wrong)
KINDS = {
    "NV_gri": dict(label="WeakLensingNvisits (gri)", unit="visits / pixel", bands=["g", "r", "i"]),
    "NV_riz": dict(label="WeakLensingNvisits (riz)", unit="visits / pixel", bands=["r", "i", "z"]),
    "EXPT": dict(label="RIZDetectionCoaddExposureTime", unit="exposure time [s]", bands=["g", "r", "i"]),
}


def metric_ebv_cut(run_name):
    # E(B-V) cut given to the metrics for this run
    if EBV_CUT_MODE == "run":
        return DUST_THRESH[run_name]
    if EBV_CUT_MODE == "fixed":
        return FIXED_LIM_EBV
    raise ValueError(f"EBV_CUT_MODE must be run or fixed, got {EBV_CUT_MODE!r}")


def sql_for(kind):
    # non-DDF visits of the selected bands, first 10 years (same structure as band_sql() of the 06/02_WL demo)
    band_clause = " or ".join("band=" + repr(b) for b in KINDS[kind]["bands"])
    return (
        "scheduler_note not like "
        + repr("DD%")
        + " and ("
        + band_clause
        + ") and night <= "
        + repr(MAX_NIGHT)
    )


def config_tag(kind, run_name):
    # Cache tag; it contains the E(B-V) cut actually used, so a map computed with another cut is never reloaded
    ebv = metric_ebv_cut(run_name)
    if kind.startswith("NV"):
        return f"{BAND}_dc{DEPTH_CUT:.2f}_exp{MIN_EXP_TIME}_ebv{ebv:.3f}_nside{NSIDE}"
    return f"exp{MIN_EXP_TIME}_ebv{ebv:.3f}_nside{NSIDE}"  # RIZDetectionCoaddExposureTime has no depth cut


pix_area = hp.nside2pixarea(NSIDE, degrees=True)
print(f"nside={NSIDE} -> npix={hp.nside2npix(NSIDE)}, pixel area = {pix_area:.4f} deg^2")
for kind in KINDS:
    print(f"SQL constraint [{kind}]:", sql_for(kind))
print(f"E(B-V) cut given to the metrics (mode {EBV_CUT_MODE}):")
for run_name, _ in RUNS_INFO:
    print(f"  {run_name:40s} ebvlim = {metric_ebv_cut(run_name):.3f}")

## 3. The metrics (signature, docstring and source, to keep the calculation traceable)

Check in particular: the names of the parameters (the 06 demo passes `min_exp_time` to `WeakLensingNvisits` and `min_expTime` to `RIZDetectionCoaddExposureTime`), the columns that are read, the order and nature of the cuts, and whether the metric needs visits in all bands (which constrains the SQL selection).

In [ ]:
for cls in (WeakLensingNvisits, RIZDetectionCoaddExposureTime):
    print("=" * 80)
    print(cls.__name__, inspect.signature(cls.__init__))
    print(inspect.getdoc(cls))
    print("-" * 80)
    print(inspect.getsource(cls.run))

## 4. Helper functions

Same helpers as in notebooks 01 and 02. Maps are numpy masked arrays (masked = pixel rejected by the metric); `healpy` ignores the mask of a masked array, so masked pixels are converted to `hp.UNSEEN` (drawn in grey) before plotting.

In [ ]:
def as_healpy(m):
    # masked array -> plain array with hp.UNSEEN in the masked pixels (drawn in grey by healpy)
    return np.ma.filled(m, hp.UNSEEN)


def run_label(run_name):
    label = f"E(B-V) < {DUST_THRESH[run_name]:.3f}"
    return label + " (baseline)" if run_name.startswith("baseline") else label


def save_fig(fig, name):
    base = join(figs_dir, name)
    fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
    fig.savefig(base + ".pdf", bbox_inches="tight")
    print("Saved:", base + ".png/.pdf")


def bundle_to_masked(bundle):
    # MetricBundle.metric_values -> float masked array (MAF badval, NaN and inf are all masked)
    data = np.ma.getdata(bundle.metric_values).astype(float)
    mask = np.ma.getmaskarray(bundle.metric_values) | ~np.isfinite(data) | (data < -600.0)
    return np.ma.masked_array(data, mask=mask)


def save_map(path, m):
    np.savez_compressed(path, data=np.ma.getdata(m), mask=np.ma.getmaskarray(m))


def load_map(path):
    with np.load(path) as f:
        return np.ma.masked_array(f["data"], mask=f["mask"])


def msum(m):
    # sum of the valid pixels (0 if all are masked)
    return float(np.ma.getdata(m)[~np.ma.getmaskarray(m)].sum())


def diff_map(map_b, map_a, fill_masked=None):
    # Pixel-by-pixel map_b - map_a.
    # fill_masked=None: keep the difference only where BOTH maps are valid (masked elsewhere).
    # fill_masked=0.0 : a masked pixel counts as 0 (0 visits, 0 s); the result is masked only where BOTH maps are masked.
    mask_b, mask_a = np.ma.getmaskarray(map_b), np.ma.getmaskarray(map_a)
    b = np.ma.getdata(map_b).astype(float)
    a = np.ma.getdata(map_a).astype(float)
    if fill_masked is None:
        return np.ma.masked_array(b - a, mask=mask_b | mask_a)
    b = np.where(mask_b, fill_masked, b)
    a = np.where(mask_a, fill_masked, a)
    return np.ma.masked_array(b - a, mask=mask_b & mask_a)


def pair_stats(map_b, map_a, dmap, with_totals=False):
    # Footprint bookkeeping and statistics of one difference map
    vb, va = ~np.ma.getmaskarray(map_b), ~np.ma.getmaskarray(map_a)
    common, gained, lost = vb & va, vb & ~va, va & ~vb
    d = dmap.compressed()
    row = {
        "area_a_deg2": va.sum() * pix_area,
        "area_b_deg2": vb.sum() * pix_area,
        "area_gained_deg2": gained.sum() * pix_area,
        "area_lost_deg2": lost.sum() * pix_area,
        "diff_mean": d.mean() if d.size else np.nan,
        "diff_median": np.median(d) if d.size else np.nan,
        "diff_std": d.std() if d.size else np.nan,
        "diff_min": d.min() if d.size else np.nan,
        "diff_max": d.max() if d.size else np.nan,
    }
    if with_totals:
        b = np.ma.getdata(map_b).astype(float)
        a = np.ma.getdata(map_a).astype(float)
        row.update(
            {
                "total_a": a[va].sum(),
                "total_b": b[vb].sum(),
                "total_diff": b[vb].sum() - a[va].sum(),
                "from_common_pixels": (b[common] - a[common]).sum(),
                "from_gained_pixels": b[gained].sum(),
                "from_lost_pixels": -a[lost].sum(),
            }
        )
    return row


def per_run_summary(maps, with_sum=False):
    rows = []
    for run_name, dust in RUNS_INFO:
        v = maps[run_name].compressed()
        rows.append(
            {
                "run": run_name,
                "E(B-V) cut of the run": dust,
                "E(B-V) cut of the metric": metric_ebv_cut(run_name),
                "n_pixels": v.size,
                "area_deg2": v.size * pix_area,
                "mean": v.mean() if v.size else np.nan,
                "median": np.median(v) if v.size else np.nan,
                "std": v.std() if v.size else np.nan,
                "min": v.min() if v.size else np.nan,
                "max": v.max() if v.size else np.nan,
            }
        )
        if with_sum:
            rows[-1]["sum"] = v.sum()
    return pd.DataFrame(rows).set_index("run")


def plot_all_maps(maps, tag, unit, title, cmap="viridis"):
    pooled = np.concatenate([m.compressed() for m in maps.values()])
    if pooled.size == 0:
        print(f"{title}: all pixels are masked in all runs, nothing to plot")
        return
    vmin, vmax = np.percentile(pooled, 1), np.percentile(pooled, 99)
    fig = plt.figure(figsize=(15, 12))
    for i, (run_name, dust) in enumerate(RUNS_INFO, start=1):
        hp.mollview(
            as_healpy(maps[run_name]),
            fig=fig.number,
            sub=(3, 3, i),
            min=vmin,
            max=vmax,
            cmap=cmap,
            title=f"{run_name}\n{run_label(run_name)}",
            unit=unit,
            cbar=True,
        )
    fig.suptitle(f"{title} (common color scale: 1st-99th percentile of all runs)", fontsize=16, y=1.02)
    save_fig(fig, f"{tag}_healpix_allruns")
    plt.show()


def plot_overlay_hist(maps, tag, xlabel, title, nbins=80):
    pooled = np.concatenate([m.compressed() for m in maps.values()])
    if pooled.size == 0:
        print(f"{title}: all pixels are masked in all runs, nothing to plot")
        return
    bins = np.linspace(np.percentile(pooled, 0.1), np.percentile(pooled, 99.9), nbins + 1)
    colors = plt.cm.viridis(np.linspace(0.0, 0.95, len(RUNS_INFO)))
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    for (run_name, dust), c in zip(RUNS_INFO, colors):
        v = maps[run_name].compressed()
        w = np.full(v.size, pix_area)
        axs[0].hist(v, bins=bins, weights=w, histtype="step", color=c, label=run_label(run_name))
        axs[1].hist(
            v, bins=bins, weights=w, histtype="step", color=c, cumulative=-1, label=run_label(run_name)
        )
    axs[0].set_ylabel("Area per bin [deg²]")
    axs[1].set_ylabel("Area with value ≥ x [deg²]")
    for ax in axs:
        ax.set_xlabel(xlabel)
        ax.grid(alpha=0.3)
    axs[0].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    save_fig(fig, f"{tag}_histograms_allruns")
    plt.show()


def plot_diff_map(dmap, run_b, run_a, tag, unit, label):
    vlim = max(np.percentile(np.abs(dmap.compressed()), 99), 1e-6) if dmap.count() else 1.0
    fig = plt.figure(figsize=(8, 5))
    hp.mollview(
        as_healpy(dmap),
        fig=fig.number,
        min=-vlim,
        max=vlim,
        cmap="RdBu_r",
        title=f"{label} difference: {run_b}\nminus {run_a}",
        unit=unit,
    )
    hp.graticule()
    save_fig(fig, f"{tag}_diff_" + f"{run_b}_MINUS_{run_a}".replace(".", "_"))
    plt.show()
    return vlim


def plot_diff_mosaic(diff_maps, tag, unit, label):
    lims = [np.percentile(np.abs(d.compressed()), 99) for d in diff_maps.values() if d.count()]
    vlim = max(max(lims), 1e-6) if lims else 1.0
    fig = plt.figure(figsize=(16, 10))
    for i, ((run_b, run_a), d) in enumerate(diff_maps.items(), start=1):
        hp.mollview(
            as_healpy(d),
            fig=fig.number,
            sub=(2, 3, i),
            min=-vlim,
            max=vlim,
            cmap="RdBu_r",
            title=f"{run_b}\nminus {run_a}",
            unit=unit,
            cbar=True,
        )
    fig.suptitle(
        f"{label}: differences between consecutive dust thresholds (common scale ±{vlim:.3g})",
        fontsize=16,
        y=1.02,
    )
    save_fig(fig, f"{tag}_diff_allpairs_combined")
    plt.show()


def plot_footprint_change(maps, tag, label):
    cmap = ListedColormap(["tab:red", "lightgreen", "tab:blue"])
    fig = plt.figure(figsize=(16, 10))
    for i, (run_b, run_a) in enumerate(PAIRS, start=1):
        vb, va = ~np.ma.getmaskarray(maps[run_b]), ~np.ma.getmaskarray(maps[run_a])
        status = np.zeros(vb.size)
        status[vb & ~va] = 1.0
        status[va & ~vb] = -1.0
        status = np.ma.masked_array(status, mask=~(va | vb))
        hp.mollview(
            as_healpy(status),
            fig=fig.number,
            sub=(2, 3, i),
            min=-1,
            max=1,
            cmap=cmap,
            cbar=False,
            title=f"{run_b}\nminus {run_a}",
        )
    fig.suptitle(
        f"{label}: footprint change between consecutive dust thresholds "
        "(blue = pixel gained, red = pixel lost, green = valid in both, dark grey = valid in neither)",
        fontsize=16,
        y=1.02,
    )
    save_fig(fig, f"{tag}_footprint_change_combined")
    plt.show()


def plot_diff_histograms(diff_maps, tag, xlabel, label):
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    for ax, ((run_b, run_a), d) in zip(axes.flat, diff_maps.items()):
        ax.hist(d.compressed(), bins=100, color="steelblue")
        ax.axvline(0, color="k", linewidth=0.8)
        ax.set_title(f"{run_b}\n- {run_a}", fontsize=9)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("N pixels")
    fig.suptitle(f"{label}: histograms of the consecutive differences")
    fig.tight_layout()
    save_fig(fig, f"{tag}_diff_histograms")
    plt.show()


def analyse_pairs(maps, tag, label, unit, xlabel, fill_masked=None, with_totals=False):
    # Difference maps (individual + mosaic), footprint change, histograms and table for all PAIRS
    diff_maps, rows = {}, []
    for run_b, run_a in PAIRS:
        print(f"=== {run_b}  -  {run_a} ===")
        d = diff_map(maps[run_b], maps[run_a], fill_masked=fill_masked)
        diff_maps[(run_b, run_a)] = d
        plot_diff_map(d, run_b, run_a, tag, unit, label)
        row = {"pair (E(B-V) thresholds)": f"{DUST_THRESH[run_b]:.3f} - {DUST_THRESH[run_a]:.3f}"}
        row.update(pair_stats(maps[run_b], maps[run_a], d, with_totals=with_totals))
        rows.append(row)
    plot_diff_mosaic(diff_maps, tag, unit, label)
    plot_footprint_change(maps, tag, label)
    plot_diff_histograms(diff_maps, tag, xlabel, label)
    stats = pd.DataFrame(rows).set_index("pair (E(B-V) thresholds)")
    stats.to_csv(join(data_dir, f"{tag}_diff_summary.csv"))
    print("Saved:", join(data_dir, f"{tag}_diff_summary.csv"))
    return diff_maps, stats

## 5. Compute (or reload) the WL maps of each run

The first execution runs MAF on the 7 databases; the maps are then cached in `data_03_WL/`. A map is recomputed only if its cache file is missing. The cache tag contains the E(B-V) cut used for the run, so changing `EBV_CUT_MODE` (or the cut of one run) never reloads a map computed with another cut.

In [ ]:
dustmap = maf_maps.DustMap(nside=NSIDE, interp=False)


def make_metric(kind, ebv_cut):
    # same calls as in 06_MAF_DESC_TaskF/02_WL_DESC_TaskForce_demo.ipynb, with the E(B-V) cut of the run
    if kind.startswith("NV"):
        return WeakLensingNvisits(
            lsst_filter=BAND,
            depth_cut=DEPTH_CUT,
            ebvlim=ebv_cut,
            min_exp_time=MIN_EXP_TIME,
            metric_name="WeakLensingNvisits_" + kind[3:],
        )
    return RIZDetectionCoaddExposureTime(
        det_bands=["r", "i", "z"],
        ebvlim=ebv_cut,
        min_expTime=MIN_EXP_TIME,
        metric_name="RIZDetectionCoaddExposureTime",
    )


def cache_path(kind, run_name):
    tag = run_name.replace(".", "_")
    return join(data_dir, f"{kind}_{tag}_{config_tag(kind, run_name)}.npz")


def load_or_run(run_name):
    ebv_cut = metric_ebv_cut(run_name)
    paths = {kind: cache_path(kind, run_name) for kind in KINDS}
    todo = [kind for kind, p in paths.items() if FORCE_RECOMPUTE or not isfile(p)]
    out = {kind: load_map(paths[kind]) for kind in KINDS if kind not in todo}
    if not todo:
        print(f"[{run_name}] loaded cached maps (ebvlim = {ebv_cut:.3f})")
        return out

    dbpath = get_db_path(run_name)
    print(f"[{run_name}] computing {todo} (ebvlim = {ebv_cut:.3f}) on {dbpath}")
    slicer = slicers.HealpixSlicer(nside=NSIDE, use_cache=False)
    bundles = {
        kind: mb.MetricBundle(
            make_metric(kind, ebv_cut),
            slicer,
            sql_for(kind),
            maps_list=[dustmap],
            run_name=run_name,
            info_label=f"{kind} {INFO_SUFFIX} ebv{ebv_cut:.3f}",
        )
        for kind in todo
    }
    group = mb.MetricBundleGroup(
        mb.make_bundles_dict_from_list(list(bundles.values())), dbpath, out_dir=data_dir, results_db=resultsDb
    )
    group.run_all()
    for kind, bundle in bundles.items():
        out[kind] = bundle_to_masked(bundle)
        save_map(paths[kind], out[kind])
        if out[kind].count() == 0:
            print(
                f"  WARNING [{run_name}] [{kind}]: all pixels are masked. Check the SQL selection and the docstring (Section 3)."
            )
    return out


results = {run_name: load_or_run(run_name) for run_name, _ in RUNS_INFO}
wl_maps = {kind: {run_name: results[run_name][kind] for run_name in RUN_NAMES} for kind in KINDS}

## 6. Per-run summary

`area_deg2` is the area of the pixels where the metric is defined (the usable WL footprint), `sum` the total over the sky (total number of visits for `NV_*`, total exposure time in seconds for `EXPT`), and `mean` / `median` / `std` / `min` / `max` describe the values over these pixels.

In [ ]:
summaries = {}
for kind, info in KINDS.items():
    summaries[kind] = per_run_summary(wl_maps[kind], with_sum=True)
    summaries[kind].to_csv(join(data_dir, f"WL_{kind}_per_run_summary.csv"))
    print(f'{info["label"]} ({info["unit"]}):')
    display(summaries[kind].round(1))

## 7. Healpix maps for the 7 footprint variants

Grey pixels are masked (rejected by the cuts of the metric).

In [ ]:
for kind, info in KINDS.items():
    plot_all_maps(wl_maps[kind], f"WL_{kind}", info["unit"], info["label"])

## 8. Histograms

Left: area-weighted histogram of the values (each pixel weighs its area). Right: area with at least a given value (reverse cumulative), the natural way to compare footprints.

In [ ]:
for kind, info in KINDS.items():
    plot_overlay_hist(
        wl_maps[kind],
        f"WL_{kind}",
        info["unit"],
        f'{info["label"]}: area-weighted histograms (left) and area with value >= x (right)',
    )

## 9. `gri` versus `riz` on the reference run

Left and middle: `WeakLensingNvisits` in `gri` and in `riz` (same color scale). Right: the ratio `riz / gri` on the pixels valid in both. Reference run: `REF_RUN` (baseline).

In [ ]:
gri, riz = wl_maps["NV_gri"][REF_RUN], wl_maps["NV_riz"][REF_RUN]
both = ~np.ma.getmaskarray(gri) & ~np.ma.getmaskarray(riz)
ratio = np.ma.masked_array(np.ma.getdata(riz) / np.maximum(np.ma.getdata(gri), 1e-12), mask=~both)

if both.sum() == 0:
    print("No pixel valid in both maps: nothing to plot")
else:
    vmax = np.percentile(np.concatenate([gri.compressed(), riz.compressed()]), 99)
    rmin, rmax = np.percentile(ratio.compressed(), [1, 99])
    fig = plt.figure(figsize=(18, 5))
    hp.mollview(
        as_healpy(gri),
        fig=fig.number,
        sub=(1, 3, 1),
        min=0,
        max=vmax,
        cmap="viridis",
        title="WeakLensingNvisits (gri)",
        unit="visits / pixel",
    )
    hp.mollview(
        as_healpy(riz),
        fig=fig.number,
        sub=(1, 3, 2),
        min=0,
        max=vmax,
        cmap="viridis",
        title="WeakLensingNvisits (riz)",
        unit="visits / pixel",
    )
    hp.mollview(
        as_healpy(ratio),
        fig=fig.number,
        sub=(1, 3, 3),
        min=rmin,
        max=rmax,
        cmap="magma",
        title="riz / gri",
        unit="ratio",
    )
    fig.suptitle(f"{REF_RUN}: WeakLensingNvisits, gri versus riz", fontsize=16, y=1.02)
    save_fig(fig, "WL_NV_gri_vs_riz_" + REF_RUN.replace(".", "_"))
    plt.show()

## 10. Differences between consecutive dust thresholds

For each pair the map is `value(run_b) - value(run_a)` with `run_b` the run with the *larger* threshold. A masked pixel counts as 0, so the map contains the pixels gained or lost with the footprint (grey = masked in both runs). In the tables, `total_diff = from_common_pixels + from_gained_pixels + from_lost_pixels`: the change of the total split into a change of the value on the pixels valid in both runs, the total of the pixels gained, and (negative) the total of the pixels lost. The footprint-change mosaic shows where the pixels are gained (blue) or lost (red).

With `EBV_CUT_MODE = 'run'` the metric cut moves with the threshold, so the gained pixels are expected to lie mostly between the two thresholds of the pair. Note that `0.199` and `baseline (0.200)` have almost the same nominal footprint: the `baseline-0.199` pair is a near-null test whose differences give the scale of the scheduler-to-scheduler fluctuations, to be compared with the other pairs.

In [ ]:
diff_results = {}
for kind, info in KINDS.items():
    print("#" * 30, info["label"])
    diff_results[kind] = analyse_pairs(
        wl_maps[kind],
        f"WL_{kind}",
        info["label"],
        f'Δ {info["unit"]}',
        f'Δ {info["unit"]}',
        fill_masked=0.0,
        with_totals=True,
    )
    display(diff_results[kind][1].round(1))

## 11. Totals and footprint area versus the dust threshold

In [ ]:
dust_values = [d for _, d in RUNS_INFO]
fig, axs = plt.subplots(1, 4, figsize=(20, 4.5))
colors = ["tab:blue", "tab:orange", "tab:purple"]
for ax, (kind, info), c in zip(axs[:3], KINDS.items(), colors):
    ax.plot(dust_values, [msum(wl_maps[kind][r]) for r in RUN_NAMES], marker="o", color=c)
    ax.set_ylabel("Total " + info["unit"].replace(" / pixel", ""))
    ax.set_title(info["label"], fontsize=10)
areas = [wl_maps["NV_gri"][r].count() * pix_area for r in RUN_NAMES]
axs[3].plot(dust_values, areas, marker="o", color="tab:green")
axs[3].set_ylabel("Footprint area [deg²]")
axs[3].set_title("WeakLensingNvisits (gri) footprint", fontsize=10)
for ax in axs:
    ax.set_xlabel("E(B-V) threshold that defines the footprint of the run")
    ax.grid(alpha=0.3)
fig.tight_layout()
save_fig(fig, "WL_totals_vs_dust_threshold")
plt.show()

## 12. `WeakLensingNvisits` versus the dust threshold, with the pixel-to-pixel dispersion

Mean and median number of visits per pixel over the WL footprint of each run, as a function of the E(B-V) threshold that defines the footprint (with `EBV_CUT_MODE = 'run'` this is also the cut of the metric), for `gri` and `riz`. Solid lines: mean; dashed lines: median. The light band is the mean ± the **standard deviation of the pixel values** inside the footprint, i.e. their dispersion over the sky, not an uncertainty on the mean (the standard error `std / sqrt(N_pixels)` would be much smaller, and neighbouring pixels are not independent). The open circle marks the baseline.

In [ ]:
x = np.array([DUST_THRESH[r] for r in RUN_NAMES])
i_ref = RUN_NAMES.index(REF_RUN)
fig, ax = plt.subplots(figsize=(8, 5.5))
for kind, color in (("NV_gri", "tab:blue"), ("NV_riz", "tab:orange")):
    vals = [wl_maps[kind][r].compressed() for r in RUN_NAMES]
    mean = np.array([v.mean() if v.size else np.nan for v in vals])
    median = np.array([np.median(v) if v.size else np.nan for v in vals])
    std = np.array([v.std() if v.size else np.nan for v in vals])
    label = KINDS[kind]["label"]
    ax.fill_between(
        x, mean - std, mean + std, color=color, alpha=0.15, linewidth=0, label=label + ": mean ± std"
    )
    ax.plot(x, mean, "o-", color=color, label=label + ": mean")
    ax.plot(x, median, "s--", color=color, label=label + ": median")
    ax.plot(
        x[i_ref], mean[i_ref], "o", mfc="none", mec="k", ms=12, label="baseline" if kind == "NV_gri" else None
    )
ax.set_xlabel("E(B-V) threshold that defines the footprint of the run")
ax.set_ylabel("WeakLensingNvisits [visits / pixel]")
ax.set_title("Mean and median over the footprint; band = mean ± pixel-to-pixel standard deviation")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "WL_NV_mean_vs_dust_threshold")
plt.show()

## 13. Notes for improving the WL metric (PSF)

What can be said from the definitions printed in Section 3 (to be checked against the source of your installation):

1. **What enters the count.** `WeakLensingNvisits` counts visits, so the PSF can only enter it through the cuts that select them, notably the coadded `i`-band depth (built from the per-visit `fiveSigmaDepth`, which contains the seeing of each visit, as discussed in notebook 01) and the minimum exposure time. If the columns read by the metric (`col=[...]` in the source) do not include a seeing column, a visit taken in poor seeing weighs exactly as much as a visit in excellent seeing, although the quality of the shear measurement depends on it.
2. **`RIZDetectionCoaddExposureTime`** is a sum of exposure times, with the same remark: an exposure time carries no information on the image quality of the visits.
3. **Hooks.** A PSF-aware version needs a seeing column of the OpSim database (for example `seeingFwhmEff`) in the `col=[...]` list, then either a weight per visit in the count, or a selection of the visits on their seeing. The seeing maps of these same runs are in `../07_variateEVmV/01_Seeing_HealpixMaps.ipynb`.
4. **Footprint.** The footprint of `WeakLensingNvisits` is a step function of the two cuts (`ebvlim`, `depth_cut`); the maps above show how much of the footprint change between two thresholds is due to each of them, through the pixels gained and lost.

## Caveats

- These are deliberately simple **proxy** metrics: more visits or more exposure time over a clean (low-extinction, deep enough) footprint is *assumed* to correlate with a better control of the weak-lensing systematics. They are meant for relative comparisons between runs, not as an absolute systematics forecast.
- With `EBV_CUT_MODE = 'run'` the metric cut follows the threshold of each run, so the metric footprint approximately reproduces the WFD footprint scheduled in that run. It is not guaranteed to be identical to it: the dust map used by MAF (`DustMap`, `nside = 64`, `interp=False`) is not necessarily the one from which the scheduler built its footprint, so pixels along the boundary may differ.
- **`EXPT` selection.** The SQL selection of `RIZDetectionCoaddExposureTime` (`gri`) is copied from the 06 demo notebook. If the docstring printed in Section 3 says that visits in all bands (or in `riz`) are required, this selection has to be changed in `KINDS`; a warning is printed when a computed map is entirely masked. The cache tag does not contain the SQL selection: use `FORCE_RECOMPUTE = True` after changing it.
- A masked pixel counts as 0 in the difference maps and totals of Sections 10-11. The `depth_cut` is fixed at the year-10 value 25.9 for all runs; another value requires to rerun MAF because the depth of the pixels is not stored in the maps.
- Two different scheduler runs do not observe the same visit sequence even where their footprint is identical, so part of the pixel-to-pixel differences is scheduling noise (see the `baseline-0.199` pair).
- The 10-year truncation applies to every run, so the numbers are not those of the full `baseline_v5.3.6_11yrs` simulation. The year-by-year evolution is not studied here (see `06_MAF_DESC_TaskF/02_WL_DESC_TaskForce_demo.ipynb`).

## References
- `rubin_sim/maf/metrics/weak_lensing_systematics_metric.py` (`WeakLensingNvisits`, `RIZDetectionCoaddExposureTime`, `ExgalM5WithCuts`): https://github.com/lsst/rubin_sim/blob/main/rubin_sim/maf/metrics/weak_lensing_systematics_metric.py
- `rubin_sim.maf.batches.science_radar_batch` (official "WL" subgroup and depth cuts per year): https://github.com/lsst/rubin_sim/blob/main/rubin_sim/maf/batches/science_radar_batch.py
- `../06_MAF_DESC_TaskF/02_WL_DESC_TaskForce_demo.ipynb` - the WL metrics on the baseline, year by year.
- `../07_variateEVmV/01_FOMNv_HealpixDiff_ShrinkFPDust.ipynb` - presentation model (maps, consecutive differences, histograms); `01_compareExgalM5withCuts.ipynb` and `02_compareGalaxyCounts.ipynb` - companion notebooks.
- Lochner, M. et al. 2018, arXiv:1808.00006, "Optimizing LSST Observing Strategy for Dark Energy Science"
- `shrink_fp_dust_*.db` simulations: https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/shrink_fp/